# 🏎️ Notebook 15 — The Great Bypass
### Direct Preference Optimization (DPO)

**Series**: RL Notebook Series · Act V — LLM Alignment · Post 15 of 17

---

## The Story So Far

In Notebook 14, we killed the **Critic** with GRPO. But we still needed a **Reward Model** to score every generated name.

The full RLHF pipeline looks like this:

```
Stage 1: SFT          →  Pre-train on good text
Stage 2: Reward Model  →  Train a separate neural network to score quality
Stage 3: RL Fine-Tune  →  PPO/GRPO to maximize scores
```

What if we could **skip Stages 2 and 3 entirely?**

### DPO: The Great Bypass

**DPO** (Direct Preference Optimization), introduced by **Rafailov et al.** at Stanford in 2023
(*"Direct Preference Optimization: Your Language Model is Secretly a Reward Model"*),
rewrites the math of RLHF so cleverly that
the reward model **disappears from the equations**. All you need is:
- Your SFT model (the frozen reference)
- A dataset of **(chosen, rejected)** pairs — human preferences
- One simple loss function

No reward model. No RL training loop. No clipped surrogates. Just classification.

## The Full Alignment Pipeline

| Step | RLHF (PPO) | GRPO | DPO |
| :--- | :---: | :---: | :---: |
| **Pre-train (SFT)** | ✅ | ✅ | ✅ |
| **Train Reward Model** | ✅ Train RM | ✅ Use RM | ❌ **Skip!** |
| **RL Training Loop** | ✅ PPO + Critic | ✅ PPO − Critic | ❌ **Skip!** |
| **Preference Data** | ❌ Not needed | ❌ Not needed | ✅ (chosen, rejected) |
| **Networks at train** | 4 (policy, ref, critic, RM) | 2 (policy, ref) | 2 (policy, ref) |
| **Complexity** | Very High | Medium | **Low** |

DPO trades the RL complexity for a simple supervised learning objective.
Let's build it!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
torch.manual_seed(42)
np.random.seed(42)

## 1. Our Tiny Language Model (Recap from NB14)

We reuse the same character-level name generator from Notebook 14.
This makes the notebook self-contained — you can run it independently.

In [ ]:
# Baby names dataset
NAMES = [
    'emma', 'olivia', 'ava', 'sophia', 'isabella', 'mia', 'charlotte', 'amelia',
    'harper', 'evelyn', 'abigail', 'emily', 'ella', 'elizabeth', 'camila', 'luna',
    'sofia', 'avery', 'mila', 'aria', 'scarlett', 'penelope', 'layla', 'chloe',
    'victoria', 'madison', 'eleanor', 'grace', 'nora', 'riley', 'zoey', 'hannah',
    'hazel', 'lily', 'ellie', 'violet', 'lillian', 'zoe', 'stella', 'aurora',
    'natalie', 'emilia', 'everly', 'leah', 'aubrey', 'willow', 'addison', 'lucy',
    'audrey', 'bella', 'nova', 'brooklyn', 'paisley', 'savannah', 'claire', 'skylar',
    'liam', 'noah', 'oliver', 'james', 'elijah', 'william', 'henry', 'lucas',
    'benjamin', 'theodore', 'jack', 'levi', 'alexander', 'mason', 'ethan', 'daniel',
    'jacob', 'michael', 'logan', 'jackson', 'sebastian', 'aiden', 'owen', 'samuel',
    'ryan', 'nathan', 'caleb', 'christian', 'dylan', 'landon', 'josiah', 'andrew',
    'thomas', 'gabriel', 'anthony', 'leo', 'lincoln', 'jaxon', 'asher', 'mateo',
    'oscar', 'ezra', 'hudson', 'miles', 'elias', 'kai', 'ace', 'nova',
    'iris', 'ivy', 'ada', 'eve', 'axel', 'cole', 'finn', 'dean',
    'jade', 'ruby', 'hope', 'eden', 'kira', 'maya', 'sara', 'alba',
    'felix', 'rex', 'max', 'hugo', 'lars', 'otto', 'rome', 'sage',
]

chars = sorted(list(set(''.join(NAMES))))
VOCAB = ['.'] + chars
VOCAB_SIZE = len(VOCAB)
stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for ch, i in stoi.items()}
CONTEXT_LEN = 3

def build_dataset(names):
    xs, ys = [], []
    for name in names:
        padded = '.' * CONTEXT_LEN + name + '.'
        for i in range(len(padded) - CONTEXT_LEN):
            context = [stoi[c] for c in padded[i:i+CONTEXT_LEN]]
            target = stoi[padded[i+CONTEXT_LEN]]
            xs.append(context)
            ys.append(target)
    return torch.tensor(xs), torch.tensor(ys)

X_train, Y_train = build_dataset(NAMES)
print(f"Vocab: {VOCAB_SIZE} chars | Training: {X_train.shape[0]} examples")

In [ ]:
class CharLM(nn.Module):
    """Tiny character-level language model (Karpathy's makemore style)."""
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64, context_len=CONTEXT_LEN):
        super().__init__()
        self.context_len = context_len
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * context_len, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        emb = self.embed(x).view(x.shape[0], -1)
        h = torch.tanh(self.fc1(emb))
        return self.fc2(h)
    
    def get_probs(self, x):
        return F.softmax(self.forward(x), dim=-1)

def pretrain_sft(model, X, Y, epochs=200, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(epochs):
        logits = model(X)
        loss = F.cross_entropy(logits, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")
    return losses

sft_model = CharLM(VOCAB_SIZE)
print("Pre-training SFT model...")
_ = pretrain_sft(sft_model, X_train, Y_train)

@torch.no_grad()
def generate_name(model, max_len=10):
    context = [0] * CONTEXT_LEN
    name = []
    for _ in range(max_len):
        x = torch.tensor([context])
        probs = model.get_probs(x)
        ix = torch.multinomial(probs[0], 1).item()
        if ix == 0: break
        name.append(itos[ix])
        context = context[1:] + [ix]
    return ''.join(name)

print("\n📝 SFT-generated names:")
for _ in range(10):
    print(f"  {generate_name(sft_model)}")

## 2. The Reward Function (For Building Preferences)

We use the same reward function from Notebook 14. But here's the twist:
**DPO doesn't use this at training time!**

We only use it to build the preference dataset *beforehand*.
Once we have the (chosen, rejected) pairs, the reward function is never called again.

In [ ]:
VOWELS = set('aeiou')
CONSONANTS = set('bcdfghjklmnpqrstvwxyz')

def reward_name(name):
    """Score a generated name. Higher = better quality."""
    if len(name) == 0:
        return -2.0
    score = 0.0
    if 3 <= len(name) <= 7:
        score += 1.0
    elif len(name) < 3:
        score -= 1.0
    elif len(name) > 8:
        score -= 0.5
    has_repeats = any(name[i] == name[i+1] == name[i+2] for i in range(len(name)-2)) if len(name) >= 3 else False
    if has_repeats:
        score -= 1.5
    if len(name) >= 2:
        alternations = sum(1 for i in range(len(name)-1) if (name[i] in VOWELS) != (name[i+1] in VOWELS))
        score += 0.5 * (alternations / (len(name) - 1))
    if name[-1] in VOWELS:
        score += 0.5
    if name[0] in CONSONANTS:
        score += 0.3
    return score

## 3. Building the Preference Dataset

This is DPO's secret ingredient. Instead of a reward model, we need **pairs** of outputs:
- **Chosen** ($y_w$): a name that humans (or our reward function) think is good
- **Rejected** ($y_l$): a name that is worse

For each "prompt" (the start-of-name token), we generate many candidates,
score them, and pair the best with the worst.

In the real world, these pairs come from human annotators comparing outputs!

In [ ]:
def build_preference_dataset(model, num_pairs=500, candidates_per_prompt=8):
    """Generate preference pairs: (chosen_name, rejected_name) based on reward scores."""
    chosen_names = []
    rejected_names = []
    
    for _ in range(num_pairs):
        # Generate a group of candidate names
        candidates = [generate_name(model) for _ in range(candidates_per_prompt)]
        # Filter out empties
        candidates = [c for c in candidates if len(c) > 0]
        if len(candidates) < 2:
            continue
        
        # Score each candidate
        scores = [(c, reward_name(c)) for c in candidates]
        scores.sort(key=lambda x: x[1])  # Sort by reward (ascending)
        
        # Chosen = highest reward, Rejected = lowest reward
        rejected_name = scores[0][0]   # Worst
        chosen_name = scores[-1][0]    # Best
        
        # Only include if there's a meaningful difference
        if scores[-1][1] > scores[0][1]:
            chosen_names.append(chosen_name)
            rejected_names.append(rejected_name)
    
    return chosen_names, rejected_names

print("Building preference dataset from SFT model...")
chosen, rejected = build_preference_dataset(sft_model, num_pairs=500)
print(f"\n✅ Created {len(chosen)} preference pairs")

print(f"\nExample pairs:")
print(f"{'Chosen':<12} {'Rew':>5}  │  {'Rejected':<12} {'Rew':>5}")
print("─" * 42)
for i in range(min(8, len(chosen))):
    print(f"{chosen[i]:<12} {reward_name(chosen[i]):>5.2f}  │  {rejected[i]:<12} {reward_name(rejected[i]):>5.2f}")

## 4. The DPO Loss Function

Here's the beautiful math. The DPO loss is:

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

Let's break this down:

| Symbol | Meaning |
| :--- | :--- |
| $\pi_\theta$ | Our **policy model** (the one we're training) |
| $\pi_{ref}$ | The **reference model** (frozen copy of the SFT model) |
| $y_w$ | The **chosen** (preferred) name |
| $y_l$ | The **rejected** (dispreferred) name |
| $\beta$ | Temperature — how much the policy can deviate from the reference |
| $\sigma$ | Sigmoid function — squashes to [0, 1] |

### Intuition

The loss function says:
1. Make the **chosen** name more likely under your policy (relative to the reference)
2. Make the **rejected** name less likely under your policy (relative to the reference)
3. But don't drift too far from the reference model ($\beta$ controls this)

If you squint, this is just **binary classification**: the model learns to assign a higher score to chosen vs rejected.
The genius is that this implicitly optimizes the same objective as RLHF — without ever training a reward model!

## 5. Computing Sequence Log-Probabilities

For DPO, we need to compute $\log \pi_\theta(y|x)$ — the total log-probability of a name under a model.
This is just the sum of log-probabilities of each character given its context.

In [ ]:
def compute_sequence_log_prob(model, name):
    """Compute the total log probability of a name under the given model.
    
    Returns the sum of log P(char_t | context_t) for each character in the name.
    """
    padded = '.' * CONTEXT_LEN + name + '.'
    total_log_prob = torch.tensor(0.0)
    
    for i in range(len(name)):
        ctx = [stoi[c] for c in padded[i:i+CONTEXT_LEN]]
        target = stoi[padded[i+CONTEXT_LEN]]
        x = torch.tensor([ctx])
        
        logits = model(x)
        log_probs = F.log_softmax(logits, dim=-1)
        total_log_prob = total_log_prob + log_probs[0, target]
    
    return total_log_prob

# Test it
test_name = chosen[0]
log_p = compute_sequence_log_prob(sft_model, test_name)
print(f"log P('{test_name}' | SFT model) = {log_p.item():.4f}")
print(f"P('{test_name}' | SFT model) = {torch.exp(log_p).item():.6f}")

## 6. DPO Training

The training loop is remarkably simple compared to PPO or GRPO:
- No rollout buffers
- No advantage estimation
- No clipping
- Just compute the loss on each (chosen, rejected) pair and backpropagate!

In [ ]:
def train_dpo(sft_model, chosen_names, rejected_names, 
              num_epochs=50, lr=1e-3, beta=0.5, batch_size=32):
    """Train a model using Direct Preference Optimization."""
    # Policy = clone of SFT (this is what we train)
    policy = CharLM(VOCAB_SIZE)
    policy.load_state_dict(sft_model.state_dict())
    
    # Reference = frozen copy of SFT
    ref_model = CharLM(VOCAB_SIZE)
    ref_model.load_state_dict(sft_model.state_dict())
    for p in ref_model.parameters():
        p.requires_grad = False
    
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    loss_history = []
    reward_history = []
    n_pairs = len(chosen_names)
    
    for epoch in range(num_epochs):
        # Shuffle the dataset
        indices = np.random.permutation(n_pairs)
        epoch_loss = 0
        num_batches = 0
        
        for start in range(0, n_pairs, batch_size):
            batch_idx = indices[start:start+batch_size]
            batch_loss = torch.tensor(0.0)
            
            for idx in batch_idx:
                w_name = chosen_names[idx]    # Chosen (winner)
                l_name = rejected_names[idx]  # Rejected (loser)
                
                # Log probs under the policy (trainable)
                log_pi_w = compute_sequence_log_prob(policy, w_name)
                log_pi_l = compute_sequence_log_prob(policy, l_name)
                
                # Log probs under the reference (frozen)
                with torch.no_grad():
                    log_ref_w = compute_sequence_log_prob(ref_model, w_name)
                    log_ref_l = compute_sequence_log_prob(ref_model, l_name)
                
                # The DPO loss! ✨
                # reward_diff = beta * [(log policy(chosen) / ref(chosen)) - (log policy(rejected) / ref(rejected))]
                log_ratio_w = log_pi_w - log_ref_w  # How much more the policy likes the chosen vs the ref
                log_ratio_l = log_pi_l - log_ref_l  # How much more the policy likes the rejected vs the ref
                
                logit = beta * (log_ratio_w - log_ratio_l)
                loss = -F.logsigmoid(logit)  # Binary cross-entropy style
                
                batch_loss = batch_loss + loss
            
            batch_loss = batch_loss / len(batch_idx)
            
            optimizer.zero_grad()
            batch_loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += batch_loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / max(num_batches, 1)
        loss_history.append(avg_loss)
        
        # Evaluate: generate names and compute average reward
        eval_rewards = [reward_name(generate_name(policy)) for _ in range(50)]
        avg_reward = np.mean(eval_rewards)
        reward_history.append(avg_reward)
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Avg Reward: {avg_reward:.3f}")
    
    return policy, loss_history, reward_history

print("Training with DPO...")
dpo_policy, dpo_losses, dpo_rewards = train_dpo(sft_model, chosen, rejected)

In [ ]:
print("\n📝 DPO-generated names:")
dpo_names = [generate_name(dpo_policy) for _ in range(15)]
for n in dpo_names:
    print(f"  '{n}' (reward: {reward_name(n):.2f})")

## 7. DPO Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(dpo_losses, color='#8b5cf6', linewidth=2)
ax.set_title('DPO Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('DPO Loss')

ax = axes[1]
ax.plot(dpo_rewards, color='#8b5cf6', linewidth=2)
ax.set_title('DPO Average Reward Over Training')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Reward')

plt.tight_layout()
plt.show()

## 8. The Full Picture: SFT vs DPO

Let's compare the reward distributions before and after DPO alignment.

In [ ]:
n_eval = 200
sft_eval = [reward_name(generate_name(sft_model)) for _ in range(n_eval)]
dpo_eval = [reward_name(generate_name(dpo_policy)) for _ in range(n_eval)]

fig, ax = plt.subplots(figsize=(10, 5))

labels = ['SFT Baseline', 'DPO Aligned']
data = [sft_eval, dpo_eval]
colors = ['#94a3b8', '#8b5cf6']

bp = ax.boxplot(data, labels=labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Reward Distribution: SFT vs DPO (200 samples each)')
ax.set_ylabel('Reward')
plt.show()

print(f"Avg Reward → SFT: {np.mean(sft_eval):.3f} | DPO: {np.mean(dpo_eval):.3f}")

print(f"\n📝 Sample SFT names:  {[generate_name(sft_model) for _ in range(8)]}")
print(f"📝 Sample DPO names:  {[generate_name(dpo_policy) for _ in range(8)]}")

## 9. ⚠️ Reward Hacking: When Alignment Goes Wrong

What happens if we push alignment **too hard**? This is one of the most important
problems in AI safety: **reward hacking** (also called **overoptimization**).

The model finds loopholes in the reward signal and exploits them,
producing outputs that score high but are clearly garbage.

Let's demonstrate this by training DPO with an **extremely high β** (making the model
deviate aggressively from the reference) and using many more epochs.

In [ ]:
# Train DPO with aggressive settings to provoke reward hacking
print("Training AGGRESSIVE DPO (high β, many epochs)...")
print("This deliberately over-optimizes to demonstrate reward hacking!\n")

hacked_policy, hacked_losses, hacked_rewards = train_dpo(
    sft_model, chosen, rejected,
    num_epochs=150,   # Much longer training
    lr=3e-3,          # Higher learning rate
    beta=5.0,         # Very high β → aggressive deviation from reference
    batch_size=16,
)

In [ ]:
# Compare: normal DPO vs hacked DPO
n_eval = 100
normal_names = [generate_name(dpo_policy) for _ in range(n_eval)]
hacked_names = [generate_name(hacked_policy) for _ in range(n_eval)]

normal_rewards = [reward_name(n) for n in normal_names]
hacked_rewards_eval = [reward_name(n) for n in hacked_names]

# Show the mode collapse
print("📝 Normal DPO names (diverse):")
for n in normal_names[:10]:
    print(f"  '{n}' (reward: {reward_name(n):.2f})")

print(f"\n📝 Over-optimized DPO names (collapsed?):")
for n in hacked_names[:10]:
    print(f"  '{n}' (reward: {reward_name(n):.2f})")

# Measure diversity
normal_unique = len(set(normal_names))
hacked_unique = len(set(hacked_names))

print(f"\n📊 Diversity check (out of {n_eval} samples):")
print(f"   Normal DPO:  {normal_unique} unique names")
print(f"   Over-opt:    {hacked_unique} unique names")
print(f"   {'⚠️ MODE COLLAPSE DETECTED!' if hacked_unique < normal_unique * 0.5 else ''}")

### Takeaway: The Alignment Tax

This is a fundamental tension in alignment:
- **Too little optimization** → The model doesn't improve much
- **Too much optimization** → The model *hacks* the reward signal, gaming the metric

This is why the **reference model** (and the β parameter) exist — they act as a leash,
preventing the policy from drifting too far from its well-behaved SFT starting point.

In real LLMs, reward hacking can look like:
- Generating repetitive, sycophantic text that always agrees with the user
- Producing verbose responses that pad length (if the reward model rewards longer text)
- Finding adversarial patterns that fool the reward model

**The KL divergence constraint is not just a regularizer — it's a safety mechanism.**

## Recap: The Full Alignment Trilogy

Over Notebooks 13 and 14, we've built the three major approaches to LLM alignment:

| | PPO (NB14) | GRPO (NB14) | DPO (NB15) |
| :--- | :--- | :--- | :--- |
| **Key Insight** | Standard RL | Kill the Critic | Kill the Reward Model |
| **Needs Reward Model** | ✅ Yes | ✅ Yes | ❌ No |
| **Needs RL Loop** | ✅ Yes | ✅ Yes | ❌ No |
| **Needs Preferences** | ❌ No | ❌ No | ✅ Yes |
| **Networks Required** | Policy + Critic + RM | Policy + RM | Policy + Reference |
| **Complexity** | Highest | Medium | **Lowest** |
| **Used By** | ChatGPT (early) | DeepSeek R1 | Llama, Zephyr, many more |

### The DPO Loss (One Last Time)

$$\mathcal{L}_{DPO} = -\log \sigma \Big( \beta \big( \underbrace{\log\pi_\theta(y_w) - \log\pi_{ref}(y_w)}_{\text{chosen gets more likely}} - \underbrace{(\log\pi_\theta(y_l) - \log\pi_{ref}(y_l))}_{\text{rejected gets less likely}} \big) \Big)$$

### What We Learned
1. **DPO** eliminates both the reward model AND the RL training loop
2. All you need is a dataset of (chosen, rejected) **preference pairs**
3. The loss is just **binary classification** — is the chosen response ranked higher?
4. A frozen **reference model** prevents the policy from drifting too far
5. DPO is equivalent to RLHF in theory, but drastically simpler in practice

---

### The Journey Continues: Act VI — Grandmasters

We've conquered LLM alignment! In the final act, we tackle the ultimate RL challenges:
**Self-Play** (AlphaZero) and **Poker** (CFR). Games so complex that the agent must become its own teacher.